#### Install and import packages

In [85]:
%pip install earthengine-api geemap
import ee
import geemap


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


#### Authenticate and initalise Earth engine project

In [86]:
ee.Authenticate()
ee.Initialize(project='editorial-democraticprimaries')

#### Russia China border geometry

In [87]:
# convert geojson to ee object
russiaChinaBorderObject  = geemap.geojson_to_ee('china-russia-border.geojson')
russiaChinaBorderGeom = russiaChinaBorderObject.geometry()
russiaChinaBorderGeomBuffered = russiaChinaBorderGeom.buffer(50000) # buffer by 50km

In [88]:
# Display border object on map
Map = geemap.Map(center=[48, 128], zoom=5)
Map.addLayer(russiaChinaBorderGeomBuffered, {}, 'Russia-China Border')
Map


Map(center=[48, 128], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…

#### Initialise nightlights dataset

In [89]:
nightlights = ee.ImageCollection('NASA/VIIRS/002/VNP46A2')
latest = nightlights.sort('system:time_start', False).first()
print('Latest image date: ' + latest.date().format().getInfo())

nightlights_filtered = nightlights.filter(ee.Filter.date('2017-01-01', latest.date())).select('Gap_Filled_DNB_BRDF_Corrected_NTL')
nighttimeVis = {min: 0.0, max: 1.0};


Latest image date: 2026-09-05T00:00:00


### Percentage nightlights change

In [90]:
# --- Scaled nightlights (no masking) ------------------------------------
# VNP46A2 NTL bands are stored with a 0.1 scale factor -> nW/cm2/sr
SCALE = 0.1

# Season filter (not a mask - it just selects which nights go into the median).
# Set SEASON = None to use the whole year; winter snow will then inflate
# radiance at this latitude.
SEASON = ee.Filter.calendarRange(5, 9, 'month')   # May-September

def to_ntl(img):
    return (img.select('Gap_Filled_DNB_BRDF_Corrected_NTL')
               .multiply(SCALE)
               .rename('NTL')
               .copyProperties(img, ['system:time_start']))

ntl = nightlights.filter(SEASON) if SEASON else nightlights
ntl = ntl.map(to_ntl)

def composite(start, end):
    return ntl.filter(ee.Filter.date(start, end)).median().rename('NTL')

In [91]:
# --- Baseline vs recent -------------------------------------------------
baseNTL   = composite('2017-01-01', '2018-01-01')
recentNTL = composite('2025-01-01', '2026-01-01')

abs_change = recentNTL.subtract(baseNTL).rename('abs_change')

# Denominator floor (nW/cm2/sr). Not a mask - nothing is hidden - it just stops
# near-zero background pixels from turning sensor noise into +/-1000%.
# Set DENOM_FLOOR = 0 for the raw ratio.
DENOM_FLOOR = 0.5

pct_change = (recentNTL.subtract(baseNTL)
                .divide(baseNTL.max(DENOM_FLOOR))
                .multiply(100)
                .rename('pct_change'))

buf = russiaChinaBorderGeomBuffered

In [92]:
# --- Map ----------------------------------------------------------------
# Log stretch on the reference layer: settlement radiance spans 3+ orders of
# magnitude, so a linear stretch shows either cities or villages, never both.
log_recent = recentNTL.max(0.05).log10()

ntlVis    = {'min': -1.3, 'max': 1.7, 'palette': ["#2c0b54",'#3b0f70','#8c2981','#de4968','#fe9f6d','#fcfdbf']}
changeVis = {'min': -100, 'max': 100, 'palette': ['#b2182b', '#f7f7f7', '#2166ac']}
absVis    = {'min': -5,   'max': 5,   'palette': ['#b2182b', '#f7f7f7', '#2166ac']}

NightlightsMap = geemap.Map(center=[48, 128], zoom=5)

# English-language basemap. The default OSM basemap labels places in the local
# script (Cyrillic / Chinese); Google's tiles honour an hl= language parameter.
NightlightsMap.clear_layers()
NightlightsMap.add_tile_layer(
    url='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}&hl=en',
    name='Google Roads (EN)',
    attribution='Google',
)

NightlightsMap.addLayer(log_recent.clip(buf),  ntlVis,    'NTL 2025 (log10)')
NightlightsMap.addLayer(abs_change.clip(buf),  absVis,    'Absolute NTL change', False)
NightlightsMap.addLayer(pct_change.clip(buf),  changeVis, '% NTL change 2017 -> 2025')
NightlightsMap.addLayer(buf, {}, 'Border buffer (50km)', False)
NightlightsMap.addLayer(russiaChinaBorderGeom, {}, 'Border', True)
NightlightsMap

Map(center=[48, 128], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…

### Populated places 


In [ ]:
# convert geojson to ee object
populatedPlacesObject  = geemap.geojson_to_ee('populated-places.geojson')
populatedPlacesGeom = populatedPlacesObject.geometry()
populatedPlacesGeomBuffered = populatedPlacesGeom.buffer(10000) # buffer by 10km